# Image -> 3D building -> SceneBundle

## ★ Cách nhanh nhất (KHÔNG cài gì): HuggingFace Space
Cài TRELLIS thủ công trên Colab hay gặp ABI hell (Pillow `_Ink`, NumPy `_center`...). Bỏ qua
hết bằng cách dùng Space chạy sẵn để xuất GLB, rồi lắp bundle trên máy CPU sạch:

1. Mở một Space, **upload ảnh tòa nhà**, bấm generate, **Download GLB**:
   - TRELLIS (Microsoft): https://huggingface.co/spaces/microsoft/TRELLIS
   - TRELLIS.2: https://huggingface.co/spaces/microsoft/TRELLIS.2
   - Hunyuan3D-2.1 (nhẹ, ổn định): https://huggingface.co/spaces/tencent/Hunyuan3D-2.1
2. Trên máy bạn (đã `pip install -r requirements.txt`), lắp `SceneBundle`:
   ```bash
   python -m builder.run --from-glb building.glb \
     --name "Tòa nhà từ ảnh" --space office \
     --prompt "Mô tả ngắn" --audience "Khách hàng mục tiêu" \
     --out frontend/public/artifacts
   ```
   Tạo `<id>.glb` + `<id>.json` (backend=generative) để demo đọc. Thuần CPU, không xung đột.

---
## (Tuỳ chọn) Tự chạy TRELLIS trên Colab GPU
Chỉ dùng nếu bạn muốn tự host. Runtime = GPU (T4/L4/A100, >=6GB VRAM). Nếu kẹt môi trường,
Runtime -> Disconnect and delete runtime để làm lại sạch, hoặc quay lại đường Space ở trên.

Notebook này KHÔNG import `builder` trên GPU và KHÔNG cài thêm deps (tránh đè môi trường TRELLIS).

In [ ]:
!nvidia-smi -L  # xác nhận có GPU

### Bước 1 - Clone TRELLIS

In [ ]:
%cd /content
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS

### Bước 2 - Cài TRELLIS theo README của họ (không cài thêm gì)

In [ ]:
%cd /content/TRELLIS
!. ./setup.sh --basic --xformers --spconv --mipgaussian --nvdiffrast
%cd /content

### Bước 3 - Upload ảnh

In [ ]:
from google.colab import files
up = files.upload()
image_path = '/content/' + list(up)[0]
print('image:', image_path)

### Bước 4 - Chạy TRELLIS -> building.glb (không import builder)

In [ ]:
import os, sys
os.environ.setdefault('ATTN_BACKEND', 'xformers')
os.environ.setdefault('SPCONV_ALGO', 'native')
sys.path.insert(0, '/content/TRELLIS')
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipe.cuda()
out = pipe.run(Image.open(image_path).convert('RGB'), seed=1)
glb = postprocessing_utils.to_glb(out['gaussian'][0], out['mesh'][0], simplify=0.95, texture_size=1024)
glb.export('building.glb')
print('saved /content/building.glb')

### Bước 5 - Tải building.glb về (rồi lắp bundle bằng lệnh --from-glb ở phần đầu)

In [ ]:
from google.colab import files
files.download('building.glb')